In [1]:
# carga de librerías

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [2]:
# Carga de datos

contract = pd.read_csv('/datasets/final_provider/contract.csv')
personal = pd.read_csv('/datasets/final_provider/personal.csv')
internet = pd.read_csv('/datasets/final_provider/internet.csv')
phone = pd.read_csv('/datasets/final_provider/phone.csv')

# Información general
print("CONTRACT")
contract.info()
print("\n")

print("PERSONAL")
personal.info()
print("\n")

print("INTERNET")
internet.info()
print("\n")

print("PHONE")
phone.info()


CONTRACT
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
dtypes: float64(1), object(7)
memory usage: 440.3+ KB


PERSONAL
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerID     7043 non-null   object
 1   gender         7043 non-null   object
 2   SeniorCitizen  7043 non-null   int64 
 3   Partner        7043 non-

In [3]:
# Revisión de valores nulos
print('Valores nulos en contract:\n', contract.isna().sum(),'\n')
print('Valores nulos en personal:\n',personal.isna().sum(),'\n')
print('Valores nulos en internet:\n',internet.isna().sum(),'\n')
print('Valores nulos en phone:\n',phone.isna().sum(),'\n')

Valores nulos en contract:
 customerID          0
BeginDate           0
EndDate             0
Type                0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
dtype: int64 

Valores nulos en personal:
 customerID       0
gender           0
SeniorCitizen    0
Partner          0
Dependents       0
dtype: int64 

Valores nulos en internet:
 customerID          0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
dtype: int64 

Valores nulos en phone:
 customerID       0
MultipleLines    0
dtype: int64 



In [4]:
# INTEGRACIÓN DE DATOS

df = contract.merge(personal, on='customerID', how='left') \
             .merge(internet, on='customerID', how='left') \
             .merge(phone, on='customerID', how='left')

df.head()

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,gender,SeniorCitizen,Partner,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,MultipleLines
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,Yes,No,DSL,No,Yes,No,No,No,No,NaN
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,No,No,DSL,Yes,No,Yes,No,No,No,No
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,No,No,DSL,Yes,Yes,No,No,No,No,No
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,No,No,DSL,Yes,No,Yes,Yes,No,No,NaN
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,No,No,Fiber optic,No,No,No,No,No,No,No


In [5]:
df['EndDate'].value_counts()

No                     5174
2019-11-01 00:00:00     485
2019-12-01 00:00:00     466
2020-01-01 00:00:00     460
2019-10-01 00:00:00     458
Name: EndDate, dtype: int64

In [6]:

df['churn'] = df['EndDate'].apply(lambda x: 0 if x == 'No' else 1)

df['churn'].value_counts()


0    5174
1    1869
Name: churn, dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   BeginDate         7043 non-null   object 
 2   EndDate           7043 non-null   object 
 3   Type              7043 non-null   object 
 4   PaperlessBilling  7043 non-null   object 
 5   PaymentMethod     7043 non-null   object 
 6   MonthlyCharges    7043 non-null   float64
 7   TotalCharges      7043 non-null   object 
 8   gender            7043 non-null   object 
 9   SeniorCitizen     7043 non-null   int64  
 10  Partner           7043 non-null   object 
 11  Dependents        7043 non-null   object 
 12  InternetService   5517 non-null   object 
 13  OnlineSecurity    5517 non-null   object 
 14  OnlineBackup      5517 non-null   object 
 15  DeviceProtection  5517 non-null   object 
 16  TechSupport       5517 non-null   object 


In [8]:
df.head()

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,gender,SeniorCitizen,...,Dependents,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,MultipleLines,churn
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85,Female,0,...,No,DSL,No,Yes,No,No,No,No,NaN,0
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5,Male,0,...,No,DSL,Yes,No,Yes,No,No,No,No,0
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15,Male,0,...,No,DSL,Yes,Yes,No,No,No,No,No,1
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75,Male,0,...,No,DSL,Yes,No,Yes,Yes,No,No,NaN,0
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65,Female,0,...,No,Fiber optic,No,No,No,No,No,No,No,1


In [9]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].isna().sum()

11

In [10]:
df.isna().sum()

customerID             0
BeginDate              0
EndDate                0
Type                   0
PaperlessBilling       0
PaymentMethod          0
MonthlyCharges         0
TotalCharges          11
gender                 0
SeniorCitizen          0
Partner                0
Dependents             0
InternetService     1526
OnlineSecurity      1526
OnlineBackup        1526
DeviceProtection    1526
TechSupport         1526
StreamingTV         1526
StreamingMovies     1526
MultipleLines        682
churn                  0
dtype: int64

In [11]:
df = df.drop(['customerID', 'EndDate', 'BeginDate'], axis=1)

In [12]:
df.columns

Index(['Type', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges',
       'TotalCharges', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'MultipleLines',
       'churn'],
      dtype='object')

In [13]:
# codificar variables categóricas

df = pd.get_dummies(df, drop_first=True)

print(df.head(3))
df.shape

   MonthlyCharges  TotalCharges  SeniorCitizen  churn  Type_One year  \
0           29.85         29.85              0      0              0   
1           56.95       1889.50              0      0              1   
2           53.85        108.15              0      1              0   

   Type_Two year  PaperlessBilling_Yes  PaymentMethod_Credit card (automatic)  \
0              0                     1                                      0   
1              0                     0                                      0   
2              0                     1                                      0   

   PaymentMethod_Electronic check  PaymentMethod_Mailed check  ...  \
0                               1                           0  ...   
1                               0                           1  ...   
2                               0                           1  ...   

   Partner_Yes  Dependents_Yes  InternetService_Fiber optic  \
0            1               0            

(7043, 21)

In [14]:
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

In [15]:
# Separar X y Y

X = df.drop('churn', axis=1)
y= df['churn']

# Dividir entrenamiento y prueba

X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y, 
                                                    test_size=0.25, 
                                                    random_state=12345, 
                                                    stratify=y)

In [16]:
df.isna().sum().sort_values(ascending=False).head(10)

MonthlyCharges                 0
Partner_Yes                    0
StreamingMovies_Yes            0
StreamingTV_Yes                0
TechSupport_Yes                0
DeviceProtection_Yes           0
OnlineBackup_Yes               0
OnlineSecurity_Yes             0
InternetService_Fiber optic    0
Dependents_Yes                 0
dtype: int64

In [17]:
y.value_counts(normalize=True)

0    0.73463
1    0.26537
Name: churn, dtype: float64

In [18]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Preparación de datos y observaciones

En esta etapa se integraron los datos provenientes de múltiples fuentes (`contract`, `personal`, `internet` y `phone`) en un único dataframe utilizando la columna `customerID` como clave de unión.

Posteriormente, se construyó la variable objetivo **`churn`**, donde:

- **0** representa clientes activos (`EndDate = "No"`)
- **1** representa clientes que cancelaron el servicio (`EndDate` contiene una fecha)

Se identificó un **desbalance moderado** en la variable objetivo, con aproximadamente:

- **73%** de clientes que permanecen  
- **27%** de clientes que cancelan  

### Preprocesamiento de datos

- La columna **`TotalCharges`** fue convertida a tipo numérico, ya que originalmente se encontraba como `object`.
- Se verificó la presencia de valores nulos, encontrando que no existen valores faltantes significativos tras la conversión.
- Se eliminaron columnas no relevantes para el modelado (**`customerID`**, **`EndDate`** y **`BeginDate`**), ya que no aportan información predictiva directa.
- Las variables categóricas fueron transformadas mediante codificación **one-hot encoding** (`get_dummies`).
- Se separaron las variables independientes (**`X`**) y la variable objetivo (**`y`**).
- Se dividieron los datos en conjuntos de entrenamiento y prueba utilizando `train_test_split`, manteniendo la proporción de clases mediante el parámetro `stratify`.

Finalmente, se aplicó un escalado de variables (**`StandardScaler`**) para preparar los datos para modelos sensibles a la escala, como la regresión logística.

Estos pasos aseguran que el dataset se encuentre limpio, estructurado y listo para el entrenamiento de modelos de clasificación.

## 4. Entrenamiento de modelos

En esta sección se entrenan diferentes modelos de clasificación para predecir la cancelación de clientes (churn). Se utilizará como métrica principal AUC-ROC, ya que es adecuada para problemas con desbalance de clases.

In [19]:
# Modelo 1: Regresión Logística

# Crear modelo
model_lr = LogisticRegression(max_iter=1000, random_state=12345)

# Entrenar modelo con datos escalados
model_lr.fit(X_train_scaled, y_train)

# Probabilidades
y_pred_proba_lr = model_lr.predict_proba(X_test_scaled)[:,1]

# AUC-ROC
auc_lr = roc_auc_score(y_test, y_pred_proba_lr)

print('AUC-ROC de Regresión Logística:', auc_lr)

AUC-ROC de Regresión Logística: 0.8282833966023386


### Resultados - Regresión Logística

El modelo de regresión logística alcanzó un AUC-ROC de 0.828, lo que indica una buena capacidad para distinguir entre clientes que cancelan y los que permanecen.

Este resultado establece una base sólida para comparar con modelos más complejos.

In [20]:
# Modelo 2: Random Forest

model_rf = RandomForestClassifier(n_estimators=100,
                                 random_state=12345)

model_rf.fit(X_train, y_train)

y_pred_proba_rf = model_rf.predict_proba(X_test)[:,1]
auc_rf = roc_auc_score(y_test, y_pred_proba_rf)

print('AUC-ROC del Bosque Aleatorio:', auc_rf)

AUC-ROC del Bosque Aleatorio: 0.8125055849928347


### Resultados - Random Forest

El modelo de Random Forest obtuvo un AUC-ROC de 0.812, ligeramente inferior al de la regresión logística.

Esto sugiere que, para este conjunto de datos, la regresión logística logra capturar mejor la relación entre las variables y la cancelación de clientes.

Por lo tanto, Random Forest no mejora el desempeño del modelo base.

In [21]:
# Modelo 3: Gradient Boosting

model_gb = GradientBoostingClassifier(random_state=12345)

model_gb.fit(X_train, y_train)

y_pred_proba_gb = model_gb.predict_proba(X_test)[:,1]

auc_gb = roc_auc_score(y_test, y_pred_proba_gb)

print('AUC-ROC de Potenciación de Gradiente:', auc_gb)

AUC-ROC de Potenciación de Gradiente: 0.8391149730761975


### Resultados - Potenciación de Gradiente

El modelo de **Potenciación de Gradiente** obtuvo un **AUC-ROC de 0.839**, el valor más alto entre los modelos entrenados hasta este punto.

Este resultado indica que el modelo tiene una mejor capacidad para distinguir entre clientes que cancelan el servicio y clientes que permanecen, en comparación con la regresión logística y el bosque aleatorio.

Por lo tanto, hasta este momento, **Potenciación de Gradiente** se considera el modelo con mejor desempeño para el problema de predicción de cancelación de clientes.

In [22]:
#Modelo 4: KNN

model_knn = KNeighborsClassifier(n_neighbors=5)

model_knn.fit(X_train_scaled, y_train)

y_pred_proba_knn = model_knn.predict_proba(X_test_scaled)[:,1]

auc_knn = roc_auc_score(y_test, y_pred_proba_knn)

print('AUC-ROC de KNN:', auc_knn)

AUC-ROC de KNN: 0.7587680250472449


### Resultados - KNN

El modelo K-Nearest Neighbors (KNN) obtuvo un AUC-ROC de 0.759, siendo el desempeño más bajo entre los modelos evaluados.

Esto puede explicarse debido a que KNN es sensible a la dimensionalidad del dataset y al uso de variables categóricas transformadas mediante one-hot encoding, lo que afecta el cálculo de distancias entre observaciones.

Por lo tanto, este modelo no resulta adecuado para este problema en comparación con otros enfoques.

In [23]:
# Modelo 5: Decision Tree

model_dt = DecisionTreeClassifier(random_state=12345)

model_dt.fit(X_train, y_train)

y_pred_proba_dt = model_dt.predict_proba(X_test)[:,1]

auc_dt = roc_auc_score(y_test, y_pred_proba_dt)

print('AUC-ROC de Árbol de Desición:', auc_dt)

AUC-ROC de Árbol de Desición: 0.645597039871057


### Resultados - Árbol de Decisión

El modelo de Árbol de Decisión obtuvo un AUC-ROC de 0.646, siendo el desempeño más bajo entre todos los modelos evaluados.

Este resultado sugiere que el modelo presenta problemas de sobreajuste, ya que los árboles de decisión tienden a ajustarse demasiado a los datos de entrenamiento y no generalizan bien a nuevos datos.

Por lo tanto, este modelo no es adecuado para este problema en comparación con otros enfoques más robustos.

## 5. Comparación de modelos y conclusión

Se entrenaron cinco modelos de clasificación para predecir la cancelación de clientes:

- Regresión Logística: AUC-ROC = 0.828  
- Random Forest: AUC-ROC = 0.812  
- Gradient Boosting: AUC-ROC = 0.839  
- KNN: AUC-ROC = 0.759  
- Árbol de Decisión: AUC-ROC = 0.646  

El modelo de **Gradient Boosting** obtuvo el mejor desempeño, alcanzando el mayor valor de AUC-ROC. Esto indica una mejor capacidad para distinguir entre clientes que cancelan el servicio y aquellos que permanecen.

La regresión logística mostró un rendimiento sólido como modelo base, mientras que Random Forest no logró superarlo. Por otro lado, KNN y el Árbol de Decisión presentaron un desempeño inferior, debido a limitaciones relacionadas con la dimensionalidad de los datos y el sobreajuste, respectivamente.

Por lo tanto, se selecciona **Gradient Boosting** como el modelo final para este problema.

In [24]:
# Optimización de hiperparámetros (Gradient Boosting)

# Se define el modelo base
gb_model = GradientBoostingClassifier(random_state=12345)

# Se definen los hiperparámetros a probar
param_grid = {'n_estimators': [50, 100, 150],
             'learning_rate': [0.05, 0.1, 0.2],
             'max_depth': [3,5]}

# Gris Search con Validación cruzada
grid_search = GridSearchCV(estimator=gb_model,
                          param_grid=param_grid,
                          scoring='roc_auc',
                          cv=5,
                          verbose=2,
                          n_jobs=-1)

# Se entrena
grid_search.fit(X_train, y_train)

# Mejores parámetros
print('Mejores parámetros:', grid_search.best_params_)
print('Mejor AUC-ROC:', grid_search.best_score_)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Mejores parámetros: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100}
Mejor AUC-ROC: 0.8463396460411218


In [25]:
best_model = grid_search.best_estimator_

y_pred_proba = best_model.predict_proba(X_test)[:, 1]

auc_final = roc_auc_score(y_test, y_pred_proba)

print("AUC-ROC final en test:", auc_final)

AUC-ROC final en test: 0.8378506961796994


## 6. Modelo final y evaluación

El modelo de Gradient Boosting fue optimizado mediante búsqueda de hiperparámetros utilizando GridSearchCV con validación cruzada de 5 folds.

Los mejores parámetros encontrados fueron:

- learning_rate = 0.05  
- max_depth = 3  
- n_estimators = 100  

El modelo optimizado alcanzó un AUC-ROC de 0.846 en validación cruzada.

Al evaluarlo en el conjunto de prueba, se obtuvo un AUC-ROC de 0.838, lo que indica una buena capacidad de generalización y ausencia de sobreajuste significativo.

En conclusión, el modelo de Gradient Boosting optimizado es la mejor solución para predecir la cancelación de clientes en este conjunto de datos.